### Chapter 6.2 - Parameter Management

Parameter management means finding, inspecting, sharing, and organizing the trainable tensors stored inside modules.


In [ ]:
from pathlib import Path
import tempfile
import torch


### 6.2.1 Parameter Access

#### 1. Intuition

A parameter is a tensor that a model learns during training.

PyTorch stores parameters inside modules and exposes them with methods such as `parameters()` and `named_parameters()`.

#### 2. Why this exists

Before debugging or customizing training, you need to know where parameters live and what shapes they have.


#### 3. Examples

Access parameter names and shapes.


In [ ]:
net = torch.nn.Sequential(
    torch.nn.Linear(2, 3),
    torch.nn.ReLU(),
    torch.nn.Linear(3, 1),
)
[(name, p.shape) for name, p in net.named_parameters()]


Access one layer's weight and bias directly.


In [ ]:
layer = net[0]
layer.weight.shape, layer.bias.shape


#### 4. Step-by-step breakdown

`named_parameters()` returns parameter names and tensors.

The names include the layer position inside `Sequential`.

`net[0]` selects the first submodule.

A `Linear` layer has a weight matrix and bias vector.

#### 5. Connection to ML systems

Parameter inspection is used in debugging, freezing layers, custom initialization, and optimizer configuration.

#### 6. Common confusion points

- Parameters are tensors wrapped for learning.
- ReLU has no parameters.
- A module can expose parameters from nested submodules.
- Shape mismatches often show up clearly in parameter inspection.


### 6.2.2 Tied Parameters

#### 1. Intuition

Tied parameters are shared by multiple parts of a model.

Sharing means two computations use the same parameter object, not just equal copied values.

#### 2. Why this exists

Parameter tying can enforce structure, reduce parameter count, or make two model parts learn together.


#### 3. Examples

Use the same layer object twice.


In [ ]:
shared = torch.nn.Linear(2, 2)
net = torch.nn.Sequential(shared, torch.nn.ReLU(), shared)
first_id = id(net[0].weight)
second_id = id(net[2].weight)
first_id == second_id


Equal object identity means actual sharing.


In [ ]:
net[0].weight is net[2].weight


#### 4. Step-by-step breakdown

`shared` is one `Linear` layer object.

The same object is inserted twice into `Sequential`.

`id(...)` returns an identity number for the object.

The two positions refer to the same weight parameter.

#### 5. Connection to ML systems

Tied parameters appear in some language models, autoencoders, and architectures where symmetry or reuse is intentional.

#### 6. Common confusion points

- Equal values are not the same as shared objects.
- Shared parameters receive gradient contributions from every use.
- Accidental sharing can create confusing behavior.
- Use `is` or `id` to check object identity.


### 6.2.3 Summary

#### 1. Intuition

Parameter management is about knowing which tensors are learned, where they are stored, and whether they are shared.

#### 2. Why this exists

This knowledge is required for initialization, freezing, saving, loading, and custom optimization.


#### 3. Examples

Count trainable parameter tensors.


In [ ]:
count = 0
for p in net.parameters():
    count += 1
count


#### 4. Step-by-step breakdown

The loop visits every parameter tensor exposed by the model.

The count measures tensors, not individual scalar values.

A weight matrix counts as one parameter tensor.

#### 5. Connection to ML systems

Real models can have thousands of parameter tensors and millions or billions of scalar parameter values.

#### 6. Common confusion points

- Tensor count and scalar count are different.
- Shared parameters should not be double-counted conceptually.
- Parameter names help locate tensors.
- Optimizers update parameters they are given.


### 6.2.4 Exercises

#### 1. Intuition

These exercises practice finding and comparing parameters.

#### 2. Why this exists

Parameter inspection is a practical debugging habit.


#### 3. Examples

Exercise 1: list parameter names.


In [ ]:
[name for name, _ in net.named_parameters()]


Exercise 2: count scalar parameter values.


In [ ]:
total = 0
for p in net.parameters():
    total += p.numel()
total


#### 4. Step-by-step breakdown

Exercise 1 checks naming.

Exercise 2 uses `numel`, which counts scalar elements inside a tensor.

The scalar count is usually what people mean by model size.

#### 5. Connection to ML systems

Counting parameters helps compare model capacity and memory use.

#### 6. Common confusion points

- `numel()` counts scalar entries.
- Names depend on module structure.
- Parameter access does not update parameters.
- Inspect before customizing optimizer behavior.
